# Supply Chain Control Tower

**Business question:** Can operational data identify late-shipment patterns, market performance differences, and supply-chain risk?

**Tools:** Python, Pandas, DuckDB SQL, KPI development, Tableau

## 1. Load and inspect

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

candidates = [Path('data/DataCoSupplyChainDataset.csv'), Path('../data/DataCoSupplyChainDataset.csv')]
DATA_PATH = next(p for p in candidates if p.exists())
df = pd.read_csv(DATA_PATH, encoding='latin1')
print('Shape:', df.shape)
df.head()

## 2. Create operational metrics

In [ ]:
df['Late_Shipment_Flag'] = (df['Days for shipping (real)'] > df['Days for shipment (scheduled)']).astype(int)
kpis = pd.DataFrame({
    'Metric': ['Unique Orders','Total Sales','Average Actual Shipping Days','Late Shipment Rate'],
    'Value': [df['Order Id'].nunique(), df['Sales'].sum(), df['Days for shipping (real)'].mean(), df['Late_Shipment_Flag'].mean()]
})
kpis

## 3. Market performance

In [ ]:
market = df.groupby('Market').agg(
    Orders=('Order Id','nunique'),
    Late_Rate=('Late_Shipment_Flag','mean'),
    Total_Sales=('Sales','sum')
).sort_values('Late_Rate', ascending=False)
market

In [ ]:
plt.figure(figsize=(8,4.5))
(market['Late_Rate']*100).plot(kind='bar')
plt.ylabel('Late shipment rate (%)')
plt.title('Late Shipment Rate by Market')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 4. SQL analysis

In [ ]:
import duckdb
con = duckdb.connect()
con.register('supply_chain', df)
con.sql('''
SELECT Market,
       COUNT(DISTINCT "Order Id") AS orders,
       AVG(Late_Shipment_Flag) AS late_rate,
       SUM(Sales) AS total_sales
FROM supply_chain
GROUP BY Market
ORDER BY late_rate DESC
''').df()

The standalone SQL version is in `SQL/supply_chain_analysis.sql`.

## 5. Tableau export

In [ ]:
tableau_columns = [
    'Order Id','Order Item Id','Delivery Status','Order Status','Late_delivery_risk','Late_Shipment_Flag',
    'Shipping Mode','order date (DateOrders)','shipping date (DateOrders)',
    'Days for shipping (real)','Days for shipment (scheduled)','Market','Order Region','Order Country',
    'Order City','Latitude','Longitude','Category Name','Product Name','Department Name',
    'Sales','Order Item Total','Order Profit Per Order','Order Item Discount','Order Item Discount Rate','Order Item Quantity'
]
tableau_df = df[tableau_columns].copy()
output_candidates = [Path('data/supply_chain_dashboard.csv'), Path('../data/supply_chain_dashboard.csv')]
output_path = output_candidates[0] if Path('data').exists() else output_candidates[1]
tableau_df.to_csv(output_path, index=False)
print('Exported:', output_path)

## 6. Limitations

- The late-shipment flag compares actual versus scheduled shipping days; it is not the same as every definition of customer delivery lateness.
- Market-level differences are descriptive and do not identify root causes.
- Tableau is used for dashboard communication; Python and SQL provide the analytical preparation.